In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader, MatchedSiphraAcquisition, match_events
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio


colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files_A = sorted((project_root/'data/260608').glob('A_SUBT_[1-4]*.csv'))
files_B = sorted((project_root/'data/260608').glob('B_SUBT_[1-4]*.csv'))
acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]
print(files_A)
print(files_B)

[PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/260608/A_SUBT_1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/260608/A_SUBT_2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/260608/A_SUBT_3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/260608/A_SUBT_4_Na22OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv')]
[PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/260608/B_SUBT_1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/260608/B_SUBT_2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/260608/B_SUBT_3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv'), Pos


========== MATCH RESULTS ==========
Events in file A:     246311
Events in file B:     253632
Matched events:       122158
Discarded from A:     124153
Discarded from B:     131474

Match fraction A: 49.60%
Match fraction B: 48.16%
Saved 122158 matched events
Output file: ../data/260609/MATCHED_Cs137.csv


In [16]:
# histograms
hists = {}

acqs = [acq_A, acq_B, acq_match]

print(acqs)


siphras = ['A', 'B', 'Summed']

for i, (siphra, acq), in enumerate(zip(siphras, acqs)):
    print(acq)
    print(siphra)
    filepath = sgnl.filepath.stem
    #src = (MetadataLoader.load(sgnl.metadataFile)).source
    src = "Cs-137"
    print(src)
    sources.append(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal, SIPHRA {siphra}", "", BITS12, 0, BITS12)
    #hist_bg = ROOT.TH1F(f"{src} Background, SIPHRA {siphra}", "", BITS12, 0 , BITS12)
    if siphra == 'Summed':
        hist_sgnl.Fill(sgnl['Combined_Summed']/32)
        #hist_bg.Fill(bg['Combined_Summed']/32)
    else:
        hist_sgnl.Fill(sgnl['s']/16)
        #hist_bg.Fill(bg['s']/16)
        #print(len(sgnl.active_chs))
    #hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    #hist_clean = hist_sgnl.Clone(f"{src} BG subtracted, SIPHRA {siphra}")
    #hist_clean.Add(hist_bg, -1)
    
    # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
    hist_sgnl.GetXaxis().SetTitle("ADC channel number")
    hist_sgnl.GetYaxis().SetTitle(r"Normalized counts")
    hists[f"{src}_SIPHRA_{siphra}"] = hist_sgnl
    #hists[f"{src}_BG_SIPHRA_{siphra}"] = hist_bg
    #hists[f"{src}_clean_SIPHRA_{siphra}"] = hist_clean
    del hist_sgnl





[SIPHRAAcquisition(File: 'A2_Cs137OnCrystalAtCenterCopperAttenuation_DoubleSiphra'), SIPHRAAcquisition(File: 'B2_Cs137OnCrystalAtCenterCopperAttenuation_DoubleSiphra'), MatchedSiphraAcquisition(File='MATCHED_Cs137')]
SIPHRAAcquisition(File: 'A2_Cs137OnCrystalAtCenterCopperAttenuation_DoubleSiphra')
A
Cs-137
SIPHRAAcquisition(File: 'B2_Cs137OnCrystalAtCenterCopperAttenuation_DoubleSiphra')
B
Cs-137
MatchedSiphraAcquisition(File='MATCHED_Cs137')
Summed
Cs-137


ValueError: Cannot retrieve data from field Combined_Summed in file A_SUBT_4_Na22OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB.csv: Usecols do not match columns, columns expected but not found: ['Combined_Summed']

In [18]:
print(hists)

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 3200, 2400)
cv.Divide(3,3)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(9)]

for i, siphra in enumerate(siphras):
    for idx, src in enumerate(sources):   
        cv.cd(3*i + idx+1)
        lg = lgds[3*i + idx]
        cv.cd(3*i + idx+1).SetLogy()

        print(src)
        print(f"{i}, {idx}")
        
        sgnl = hists[f"{src}_SIPHRA_{siphra}"] 
        #bg = hists[f"{src}_BG_SIPHRA_{siphra}"] 
        #clean = hists[f"{src}_clean_SIPHRA_{siphra}"]
        lg.AddEntry(sgnl, "Signal")
        #lg.AddEntry(bg, "Background")
        #lg.AddEntry(clean, "Subtracted")
        sgnl.SetLineColor(colors[0])
        #bg.SetLineColor(colors[1])
        #clean.SetLineColor(colors[2])
        if siphra == "Summed":
            sgnl.SetTitle(f"{src}, Summed detector A and B")
        else:
            sgnl.SetTitle(f"{src}, Detector setup {siphra}")
        sgnl.Draw("hist")
        #bg.Draw("sames hist")
        #clean.Draw("sames hist")
        ROOT.gPad.Update()
        #for j, sp in enumerate([sgnl, bg, clean]):
        #    st = sp.FindObject("stats")
            # print(type(st))
        #    st.SetY1NDC(Yinit - j*0.08)
        #    st.SetY2NDC(0.1)
        lg.Draw()
        
        cv.Draw()

{'Cs-137_SIPHRA_A': <cppyy.gbl.TH1F object at 0x639b4aa842e0>, 'Cs-137_SIPHRA_B': <cppyy.gbl.TH1F object at 0x639b4aa803d0>}
Cs-137 (mainly)
0, 0


KeyError: 'Cs-137 (mainly)_SIPHRA_A'

In [ ]:

file_names = ["MATCHED_SUBT_1_BackgroundAtStart_DoubleSiphra_DisableCh15SiphraB", "MATCHED_SUBT_2_Cs137OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB",
              "MATCHED_SUBT_3_Co60OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB", "MATCHED_SUBT_4_Na22OnCrystalAtCenter_DoubleSiphra_DisableCh15SiphraB"]

#file_names = ["MATCHED_SUBT_29_DoubleSiphra_BackgroundOnly_5e6events","MATCHED_SUBT_30_DoubleSiphra_Cs137OnCrystalNearCenter_5e6events"]

#ONLY RUN THIS ONCE!
for i, (file_B, file_A) in enumerate(zip(files_B, files_A)):
    match_events(file_B, file_A, output_path=f"../data/260608/{file_names[i]}.csv", tolerance = 500)

files_matched = sorted((project_root/'data/260608').glob('MATCHED_SUBT*.csv'))
acqs_matched = [MatchedSiphraAcquisition(file) for file in files_matched]
print(acqs_matched)

In [ ]:
# histograms
hists = {}

acqs = [acqs_A, acqs_B, acqs_matched]

print(acqs)


siphras = ['A', 'B', 'Summed']

for i, (siphra, acq), in enumerate(zip(siphras, acqs)):
    bg = acq[0]
    sources = []

    for sgnl in acq[1:]:
        filepath = sgnl.filepath.stem
        src = (MetadataLoader.load(sgnl.metadataFile)).source
        #src = "Cs-137"
        print(src)
        sources.append(src)
        # Signal and Background
        hist_sgnl = ROOT.TH1F(f"{src} Signal, SIPHRA {siphra}", "", BITS12, 0, BITS12)
        hist_bg = ROOT.TH1F(f"{src} Background, SIPHRA {siphra}", "", BITS12, 0 , BITS12)
        if siphra == 'Summed':
            hist_sgnl.Fill(sgnl['Combined_Summed']/32)
            hist_bg.Fill(bg['Combined_Summed']/32)
        elif siphra == 'B':
            hist_sgnl.Fill(sgnl['s']/15)
            hist_bg.Fill(bg['s']/15)
        else:
            hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
            hist_bg.Fill(bg['s']/len(bg.active_chs))
            print(len(sgnl.active_chs))
        hist_bg.Scale(sgnl.exposure/bg.exposure)
        # Clean spectrum
        hist_clean = hist_sgnl.Clone(f"{src} BG subtracted, SIPHRA {siphra}")
        hist_clean.Add(hist_bg, -1)
        for hist in [hist_sgnl, hist_bg, hist_clean]:
            # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
            hist.GetXaxis().SetTitle("ADC channel number")
            hist.GetYaxis().SetTitle(r"Normalized counts")
        hists[f"{src}_SIPHRA_{siphra}"] = hist_sgnl
        hists[f"{src}_BG_SIPHRA_{siphra}"] = hist_bg
        hists[f"{src}_clean_SIPHRA_{siphra}"] = hist_clean
        del hist_sgnl, hist_bg

In [ ]:
# histograms
hists = {}

acqs = [acqs_A, acqs_B, acqs_matched]

print(acqs)


siphras = ['A', 'B', 'Summed']

for i, (siphra, acq), in enumerate(zip(siphras, acqs)):
    bg = acq[0]
    sources = []

    for sgnl in acq[1:]:
        filepath = sgnl.filepath.stem
        src = (MetadataLoader.load(sgnl.metadataFile)).source
        #src = "Cs-137"
        print(src)
        sources.append(src)
        # Signal and Background
        hist_sgnl = ROOT.TH1F(f"{src} Signal, SIPHRA {siphra}", "", BITS12, 0, BITS12)
        hist_bg = ROOT.TH1F(f"{src} Background, SIPHRA {siphra}", "", BITS12, 0 , BITS12)
        if siphra == 'Summed':
            hist_sgnl.Fill(sgnl['Combined_Summed']/32)
            hist_bg.Fill(bg['Combined_Summed']/32)
        elif siphra == 'B':
            hist_sgnl.Fill(sgnl['s']/15)
            hist_bg.Fill(bg['s']/15)
        else:
            hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
            hist_bg.Fill(bg['s']/len(bg.active_chs))
            print(len(sgnl.active_chs))
        hist_bg.Scale(sgnl.exposure/bg.exposure)
        # Clean spectrum
        hist_clean = hist_sgnl.Clone(f"{src} BG subtracted, SIPHRA {siphra}")
        hist_clean.Add(hist_bg, -1)
        for hist in [hist_sgnl, hist_bg, hist_clean]:
            # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
            hist.GetXaxis().SetTitle("ADC channel number")
            hist.GetYaxis().SetTitle(r"Normalized counts")
        hists[f"{src}_SIPHRA_{siphra}"] = hist_sgnl
        hists[f"{src}_BG_SIPHRA_{siphra}"] = hist_bg
        hists[f"{src}_clean_SIPHRA_{siphra}"] = hist_clean
        del hist_sgnl, hist_bg

In [ ]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 3200, 2400)
cv.Divide(3,3)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(9)]

for i, siphra in enumerate(siphras):
    for idx, src in enumerate(sources):   
        cv.cd(3*i + idx+1)
        lg = lgds[3*i + idx]
        cv.cd(3*i + idx+1).SetLogy()

        print(src)
        print(f"{i}, {idx}")
        
        sgnl = hists[f"{src}_SIPHRA_{siphra}"] 
        bg = hists[f"{src}_BG_SIPHRA_{siphra}"] 
        clean = hists[f"{src}_clean_SIPHRA_{siphra}"]
        lg.AddEntry(sgnl, "Signal")
        lg.AddEntry(bg, "Background")
        lg.AddEntry(clean, "Subtracted")
        sgnl.SetLineColor(colors[0])
        bg.SetLineColor(colors[1])
        clean.SetLineColor(colors[2])
        if siphra == "Summed":
            sgnl.SetTitle(f"{src}, Summed detector A and B")
        else:
            sgnl.SetTitle(f"{src}, Detector setup {siphra}")
        sgnl.Draw("hist")
        bg.Draw("sames hist")
        clean.Draw("sames hist")
        ROOT.gPad.Update()
        for j, sp in enumerate([sgnl, bg, clean]):
            st = sp.FindObject("stats")
            # print(type(st))
            st.SetY1NDC(Yinit - j*0.08)
            st.SetY2NDC(0.1)
        lg.Draw()
        
        cv.Draw()

In [ ]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum
siphra = "Summed"
src = "Na-22"

hist = hists[f"{src}_clean_SIPHRA_{siphra}"]
energy_ranges = [(80,150), (280, 340), (390, 440)]
energies = NA22_MEV

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

In [ ]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][0], BITS12, title = r"^{137}Cs 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][1], BITS12, title = r"^{137}Cs 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
#hist_cal_bg_Cs137.SetTitle(r" ^{137}Cs, 2-sided setup detector A")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.8)])
print(res_Cs137)



In [ ]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][0], BITS12, title = r"^{137}Cs 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][1], BITS12, title = r"^{137}Cs 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.85)])
print(res_Cs137)


In [ ]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][0], BITS12, title = r"^{137}Cs 2-sided setup summed", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][1], BITS12, title = r"^{137}Cs 2-sided setup summed", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.75)])
print(res_Cs137)


In [ ]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][0], BITS12, title = r"^{22}Na 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][2], BITS12, title = r"^{22}Na 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, ((0.35, 0.65), (1, 1.45), (1.6, 1.8)))
print(res_Cs137)


In [ ]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][0], BITS12, title = r"^{22}Na 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][2], BITS12, title = r"^{22}Na 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, ((0.36, 0.65), (1.1, 1.6), (1.7,2.0)))
print(res_Cs137)

In [ ]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][0], BITS12, title = r"^{22}Na 2-sided setup summed A and B", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][2], BITS12, title = r"^{22}Na 2-sided setup summed A and B", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.36, 0.62), (1.15, 1.45), (1.63, 1.9)])
print(res_Cs137)

In [ ]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][0], BITS12, title = r"^{60}Co 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][3], BITS12, title = r"^{60}Co 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][3].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(2.1, 2.5)])
print(res_Cs137)

In [ ]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][0], BITS12, title = r"^{60}Co 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][3], BITS12, title = r"^{60}Co 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][3].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(2.25, 2.67)])
print(res_Cs137)

In [ ]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][0], BITS12, title = r"^{60}Co 2-sided setup summed A and B", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][3], BITS12, title = r"^{60}Co 2-sided setup summed A and B", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][3].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(2.28, 2.5)])
print(res_Cs137)